# Hierarchical Risk Parity, end to end

_Allocate without inverting a covariance matrix._

**pyportfolios.com** · [/research/hierarchical-risk-parity](https://pyportfolios.com/research/hierarchical-risk-parity)

> Runs on numpy + pandas + scipy + matplotlib. Synthetic, seeded data — illustrative, not investment advice.

## HRP in ~40 lines

In [ ]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

def _ivp(cov):
    ivp = 1 / np.diag(cov)
    return ivp / ivp.sum()

def _cluster_var(cov, items):
    c = cov.loc[items, items]
    w = _ivp(c)                          # 1-D inverse-variance weights
    return float(w @ c.values @ w)       # wᵀ Σ w as a scalar

def _quasi_diag(link):
    link = link.astype(int)
    order = pd.Series([link[-1, 0], link[-1, 1]])
    n = link[-1, 3]
    while order.max() >= n:
        order.index = range(0, order.shape[0] * 2, 2)
        clusters = order[order >= n]
        i, j = clusters.index, clusters.values - n
        order[i] = link[j, 0]
        order = pd.concat([order, pd.Series(link[j, 1], index=i + 1)])
        order = order.sort_index().reset_index(drop=True)
    return order.tolist()

def hrp(returns):
    cov, corr = returns.cov(), returns.corr()
    dist = np.sqrt((1 - corr) / 2.0)
    link = linkage(squareform(dist, checks=False), "single")
    order = corr.index[_quasi_diag(link)].tolist()
    w = pd.Series(1.0, index=order)
    clusters = [order]
    while clusters:
        clusters = [c[s:e] for c in clusters
                    for s, e in ((0, len(c) // 2), (len(c) // 2, len(c)))
                    if len(c) > 1]
        for i in range(0, len(clusters), 2):
            c0, c1 = clusters[i], clusters[i + 1]
            v0, v1 = _cluster_var(cov, c0), _cluster_var(cov, c1)
            alpha = 1 - v0 / (v0 + v1)
            w[c0] *= alpha
            w[c1] *= 1 - alpha
    return w.sort_index()

## Run it on three correlated blocks

In [ ]:
rng = np.random.default_rng(1)
def block(n, rho, T):
    z = rng.normal(size=(T, 1)); e = rng.normal(size=(T, n))
    return np.sqrt(rho) * z + np.sqrt(1 - rho) * e

T = 600
R = np.hstack([block(4, 0.7, T), block(4, 0.5, T), block(2, 0.3, T)]) / 100
returns = pd.DataFrame(R, columns=[f"A{i:02d}" for i in range(R.shape[1])])

w = hrp(returns)
print(w.round(3))
print(f"\nlargest weight: {w.max():.1%}   effective N: {1 / (w ** 2).sum():.1f}")